In [ ]:
%%writefile program.cpp
#include <iostream>
#include <queue>
#include <omp.h>

using namespace std;

// ?? Tree Node
struct Node {
    int data;
    Node* left;
    Node* right;

    Node(int val) {
        data = val;
        left = right = NULL;
    }
};

// ?? Parallel BFS (Level Order)
void parallelBFS(Node* root) {
    if (!root) return;

    queue<Node*> q;
    q.push(root);

    cout << "Parallel BFS: ";

    while (!q.empty()) {
        int size = q.size();

        // Process nodes of same level in parallel
        #pragma omp parallel for
        for (int i = 0; i < size; i++) {

            Node* current;

            #pragma omp critical
            {
                current = q.front();
                q.pop();
                cout << current->data << " ";
            }

            // Add children
            #pragma omp critical
            {
                if (current->left) q.push(current->left);
                if (current->right) q.push(current->right);
            }
        }
    }
    cout << endl;
}

// ?? CORRECT DFS (Sequential for proper order)
void dfs(Node* root) {
    if (!root) return;

    cout << root->data << " ";
    dfs(root->left);
    dfs(root->right);
}

// ?? Wrapper
void runDFS(Node* root) {
    cout << "DFS : ";

    #pragma omp parallel
    {
        #pragma omp single
        {
            dfs(root);  // sequential DFS
        }
    }

    cout << endl;
}

// ?? Main
int main() {

    /*
            1
          /   \
         2     3
        / \   / \
       4   5 6   7
    */

    Node* root = new Node(1);
    root->left = new Node(2);
    root->right = new Node(3);
    root->left->left = new Node(4);
    root->left->right = new Node(5);
    root->right->left = new Node(6);
    root->right->right = new Node(7);

    parallelBFS(root);
    runDFS(root);

    return 0;
}

!g++ program.cpp -o program

!./program